This file was for trying different ways to find the LSD using lat lon, landed on the shapely technique with the LSD grid. Searching for the coordinate inside the grid, and returning the lsd with sec twn rng mer formatted

In [32]:
import geopandas as gpd
from shapely.geometry import Point
import pandas as pd

In [33]:
ab_emission = pd.read_csv("ab_emissions.csv")

In [41]:
ab_emission.index.size

6111

Finds the LSD based on coordinates, but really slow

In [40]:
def find_dls_from_polygons(row, twp_grid, lsd_grid):
    lat = row['Latitude']
    lon = row['Longitude']
    
    # Ensures the coordinate reference system of the shp file is in espg 4326
    if twp_grid.crs.to_epsg() != 4326:
        twp_grid = twp_grid.to_crs(epsg=4326)
    
    if lsd_grid.crs.to_epsg() != 4326:
        lsd_grid = lsd_grid.to_crs(epsg=4326)

    point = Point(lon, lat)
    twp_polygon = twp_grid[twp_grid.contains(point)].union_all()
    shape_filtered = lsd_grid[lsd_grid.intersects(twp_polygon)]
    result = shape_filtered[shape_filtered.contains(point)]

    if result.empty:
        return None
    
    lsd = str(result['LS'].iloc[0]).zfill(2)
    sec = str(result['SEC'].iloc[0]).zfill(2)
    twp = str(result['TWP'].iloc[0]).zfill(3)
    rge = str(result['RGE'].iloc[0]).zfill(2)
    mer = "W" + str(result['M'].iloc[0]) + "M"

    dls_string = f"{lsd}-{sec}-{twp}-{rge}-{mer}"
        
    return dls_string


In [35]:
shps_path = "ATS_Polygons_SHP_Geographic/V4-1_"
twp_path = shps_path + "TWP.shp"
lsd_path = shps_path + "LSD.shp"

twp_grid = gpd.read_file(twp_path)
lsd_grid = gpd.read_file(lsd_path)

# Apply the function to each row in the dataframe
ab_emission['DLS'] = ab_emission.head(1).apply(find_dls_from_polygons, axis=1, twp_grid=twp_grid, lsd_grid=lsd_grid)

C:\Users\gusta\AppData\Local\Temp\ipykernel_32148\1614173327.py:13: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  twp_polygon = twp_grid[twp_grid.contains(point)].unary_union


In [39]:
ab_emission['DLS'].head(1).iloc[0]

'11-17-56-21-W4M'